# Configurando bibliotecas, Dataset e fazendo conexão com W&B


In [ ]:
!pip install -r requirements.txt

In [ ]:
import wandb
import pandas as pd
import seaborn as sns
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp , chi2_contingency
import yaml , random, matplotlib.pyplot as plt
import kagglehub , shutil , os
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

## Baixando base de dados

In [ ]:
# Download do dataset
path = kagglehub.dataset_download("fedesoriano/stellar-classification-dataset-sdss17")
print(f"Downloaded to: {path}")

# Copia o dataset para a pasta destinada
destination = os.getcwd ()
shutil.copytree(path , os.path.join(destination , "data/"), dirs_exist_ok=True)

# Carregar aquivo principal de acidentes
df_raw = pd.read_csv("data/star_classification.csv", low_memory=False)

print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns) }...")

## Login no W&B

In [ ]:
import key

API_KEY = key.KEY 
wandb.login(key=API_KEY)

## Registrar dados brutos W&B

In [ ]:
wandb.init(project="star-classification", job_type="load_raw", name="load_raw")
artifact = wandb.Artifact("raw_data", type="dataset", description="star_classification raw dataset from Kaggle")
temp_path = "data/temp_raw.csv"
df_raw.to_csv(temp_path , index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)
wandb.finish()
print("Artefato raw_data salvo no W&B")

"""O procedimento load_raw consiste em registrar os dados brutos como um artefato versionado
 no Weights & Biases imediatamente após a ingestão, garantindo a rastreabilidade"""

# Pré-processamento e Limpeza dos Dados

## Criando arquivo de limpesa

In [ ]:
df_raw = pd.read_csv("data/star_classification.csv", sep=",")

#Criando um arquivo para limpeza
df_raw.to_csv("data/temp_clean.csv", index = False)
df_clean = pd.read_csv("data/temp_clean.csv", sep=",")

## Visialização de dados

In [ ]:
# Dataset principal não possui dados faltantes
df_raw.info() 

# Temos dados u, g e z têm valores de -9999.000000 que são considerados nulos na astronomia
df_raw.describe()

## Tratamento de outliers

A identificação do outlier foi realizada com base na análise visual por Boxplot, porém não foi aplicado diretamente o método do IQR, uma vez que grande parte dos dados correspondia a valores reais. Optou-se pela remoção de um único registro discrepante, que apresentava valor nulo representado por -9999.000000. Essa exclusão fundamenta-se na melhoria da qualidade da base ("Garbage in, garbage out"), prevenindo que esse valor artificialmente extremo domine o aprendizado ou gere gradientes excessivamente grandes e instáveis durante o gradiente descendente da MLP. O tratamento adotado assegura maior estabilidade ao otimizador (como o Adam), facilitando a convergência para o mínimo da função de perda e promovendo melhor generalização do modelo final.

### Observando outlier com gráficos boxplot

In [ ]:
# Gráfico para o Filtro Ultravioleta (u)
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_raw['u'], color="skyblue")
plt.title('Identificação de Outliers: Filtro Ultravioleta (u)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xlabel('Magnitude u')
plt.show()

# Gráfico para o Filtro Verde (g)
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_raw['g'], color="lightgreen")
plt.title('Identificação de Outliers: Filtro Verde (g)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xlabel('Magnitude g')
plt.show()

# Gráfico para o Filtro Infravermelho (z)
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_raw['z'], color="salmon")
plt.title('Identificação de Outliers: Filtro Infravermelho (z)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.xlabel('Magnitude z')
plt.show()

### Buscando e dado com valores nulos

In [ ]:
df_raw[df_raw['u'] <= 0]['u']

In [ ]:
df_raw[df_raw['g'] <= 0]['g']

In [ ]:
df_raw[df_raw['z'] <= 0]['z']

### Removendo dado com valor nulo

In [ ]:
# Removendo o valor de index 79543 que possui dados nulos
df_clean.drop(index=79543, inplace=True)

# Salvando alterações 
df_clean.to_csv('data/temp_clean.csv', index=False)

df_clean.describe()

## Remoção de colunas irrelevantes


In [ ]:
# Eliminando colunas irrelevantes 
df_clean.drop(['obj_ID','run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'fiber_ID', 'plate', 'MJD'], axis=1, inplace=True)
# Salvando alterações no arquivo temporário
df_clean.to_csv('data/temp_clean.csv', index=False)

## Análize de dados


In [ ]:
#Gráfico de Barras
cont = df_clean['class'].value_counts()
plt.bar(cont.index, cont.values)
plt.xlabel('Classe')
plt.ylabel('Contagem')
plt.title('Gráfico de Classes')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Histograma das distribuição do todo em cada caracteristica
df_clean.hist(column=['alpha', 'delta', 'u', 'g', 'r', 'i', 'z','redshift'], bins=50, figsize=(12,12))
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Gráfico de Dispersão da distribuição dos corpos de acordo com as coordenadas na esfera celeste
plt.scatter(df_clean['alpha'], df_clean['delta'], color="blue", alpha=0.1)
plt.xlabel('Alpha')
plt.ylabel('Delta')
plt.title('Distribuição espacial')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Mapa de calor 
cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift'] # Dados relevantes com caracteristicas físicas
corr_matrix = df_clean[cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, 
    annot=True,      # Mostra os valores de 'r' em cada célula
    fmt='.2f', 
    cmap='coolwarm', # Azul para negativo, vermelho para positivo
    vmin=0.0, vmax=1.0, # Escala padrão da correlação de Pearson
    linewidths=0.5, 
    square=True
)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()



### Registrar dados limpos W&B


In [ ]:
wandb.init(project ="star-classification",job_type ="clean_data",name ="clean_data")
artifact = wandb.Artifact("clean_data",type = "dataset", description ="star_classification after deduplication and imputation")
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_clean)
wandb.summary["dropped_columns"] = df_clean.shape[1]
wandb.finish()

## Divisão estratificada de treino/teste

In [ ]:
def split_train_test(df: pd.DataFrame, target_col: str = "class", test_size: float = 0.2, random_state: int = 42):
    x = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        x, y, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=y # Mantém a proporção de GALAXY, STAR e QSO [2]
    )

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)
    return train_df, test_df

# Supondo que df_clean seja o seu dataframe carregado de star_classification.csv
train_df, test_df = split_train_test(df_clean, target_col = "class" , test_size = 0.2, random_state = 42)


### Comparação da Distribuição Estatística

In [ ]:
def compare_distributions(train_df: pd.DataFrame, test_df: pd.DataFrame, columns: list) -> dict:
    results = {}
    for col in columns:
        train_vals = train_df[col].dropna()
        test_vals = test_df[col].dropna()

        # Verifica se é numérico para teste KS ou categórico para Chi2 [3]
        if train_df[col].dtype in ["int64", "float64"]:
            stat, p = ks_2samp(train_vals, test_vals)
            results[col] = {"test": "KS", "statistic": stat, "p_value": p}
        else:
            train_counts = train_vals.value_counts(normalize=True)
            test_counts = test_vals.value_counts(normalize=True)
            all_cats = sorted(set(train_counts.index).union(test_counts.index))
            train_probs = [train_counts.get(c, 0) for c in all_cats]
            test_probs = [test_counts.get(c, 0) for c in all_cats]
            chi2, p, _, _ = chi2_contingency([train_probs, test_probs])
            results[col] = {"test": "Chi2", "statistic": chi2, "p_value": p}
    return results

feature_cols = [c for c in train_df.columns if c != "class"]
comp_results = compare_distributions(train_df, test_df, feature_cols)

### Divisões de registro e tabela de comparação com W&B

In [ ]:
wandb.init(project="star-classification", job_type="split_data", name="split_data")

# Log de artefatos (Datasets versionados) [4]
for split_name, split_df in [("train_data", train_df), ("test_data", test_df)]:
    path = f"data/temp_{split_name}.csv"
    split_df.to_csv(path, index=False)
    art = wandb.Artifact(split_name, type="dataset")
    art.add_file(path)
    wandb.log_artifact(art)

# Log da tabela de comparação estatística [5]
comp_df = pd.DataFrame(comp_results).T.reset_index()
comp_df.columns = ["feature", "test", "statistic", "p_value"]
comp_table = wandb.Table(dataframe=comp_df)
wandb.log({"distribution_comparison": comp_table})

wandb.summary["train_size"] = len(train_df)
wandb.summary["test_size"] = len(test_df)
wandb.finish()

print("Split concluído e artefatos salvos no W&B.")

## Normalização e Padronização:

In [ ]:
train_df = pd.read_csv("data/temp_train_data.csv")
test_df = pd.read_csv("data/temp_test_data.csv")

# 1. Definir os grupos de colunas
cols_minmax = ['alpha', 'delta']
cols_standard = ['u', 'g', 'r', 'i', 'z']
cols_robust = ['redshift']

# 2. Inicializar os scalers
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()
scaler_robust = RobustScaler()

# 3. Ajustar (fit) APENAS no treino e transformar ambos
# Coordenadas Angulares
train_df[cols_minmax] = scaler_minmax.fit_transform(train_df[cols_minmax])
test_df[cols_minmax] = scaler_minmax.transform(test_df[cols_minmax])

# Magnitudes
train_df[cols_standard] = scaler_standard.fit_transform(train_df[cols_standard])
test_df[cols_standard] = scaler_standard.transform(test_df[cols_standard])

# Redshift (Resistente a outliers)
train_df[cols_robust] = scaler_robust.fit_transform(train_df[cols_robust])
test_df[cols_robust] = scaler_robust.transform(test_df[cols_robust])

# verificar onde tem que guardar isso e se tem que criar artefado W&B

# Seleção de Variáveis (Feature Selection)

Aplicar três métodos distintos (ex: Correlação, Informação Mútua e Importância por Random Forest) para criar um ranking final de importância.